# **Random Meal Planner**

In [5]:
import json
import os
import sys

sys.path.insert(0, os.path.abspath(".."))

from datetime import datetime, timedelta
from random import randint, random

from engines import RandomMealPlanner

from models import (
    Ingredient,
    MealPlanningEnvironment,
    NutritionalInformation,
    Pantry,
    PantryIngredient,
    UserPreferences,
)

In [6]:
preferences = UserPreferences(
    weekly_budget=50,
    calorie_target_per_day=2500,
    protein_target_per_day=80,
    is_vegetarian=False,
    is_vegan=False,
    requires_gluten_free=False,
    requires_lactose_free=False,
)

In [7]:
all_ingredients = []

with open("../recipe_extraction/supplemented_structured_ingredients.json", "r") as f:
    ingredients_data = json.load(f)

    for ingredient_data in ingredients_data:
        ingredient_nutritional_information = NutritionalInformation(**ingredient_data["nutritional_information"])

        ingredient = Ingredient(
            name=ingredient_data["name"], nutritional_information=ingredient_nutritional_information
        )
        all_ingredients.append(ingredient)

In [8]:
def get_ingredient(ingredient_name: str) -> Ingredient | None:
    return next((ingredient for ingredient in all_ingredients if ingredient.name == ingredient_name), None)

In [9]:
def get_pantry_ingredient(
    ingredient_name: str, estimated_expiration_date: datetime, estimated_financial_cost: float
) -> PantryIngredient:
    ingredient = get_ingredient(ingredient_name)

    if ingredient is None:
        raise ValueError(f"Ingredient '{ingredient_name}' not found in ingredient list")

    return PantryIngredient(
        name=ingredient.name,
        nutritional_information=ingredient.nutritional_information,
        estimated_expiration_date=estimated_expiration_date,
        estimated_financial_cost=estimated_financial_cost,
    )

In [10]:
CURRENT_DATE = datetime.now()

In [11]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [12]:
pantry_ingredients = [
    get_pantry_ingredient(ingredient_name, CURRENT_DATE + timedelta(days=randint(1, 7)), random())
    for ingredient_name in pantry_ingredient_name_by_quantity.keys()
]

In [13]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [14]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [15]:
pantry.print()

---
Quantity: 800 g
Ingredient: chicken breast
	Estimated Expiration Date: 2026-04-30
	Estimated Financial Cost per 100g: EUR 0.35
	Nutritional Information:
		Calories: 125.0 kcal
		Carbohydrates: 1.79 g
		Sugar: 0.0 g
		Protein: 16.07 g
		Fat: 5.36 g
		Saturated Fat: 1.79 g
		Fiber: 1.8 g
		Sodium: 571.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: No
		Vegan: No
---
---
Quantity: 1500 g
Ingredient: broccoli
	Estimated Expiration Date: 2026-04-26
	Estimated Financial Cost per 100g: EUR 0.98
	Nutritional Information:
		Calories: 31.0 kcal
		Carbohydrates: 6.27 g
		Sugar: 1.4 g
		Protein: 2.57 g
		Fat: 0.34 g
		Saturated Fat: 0.039 g
		Fiber: 2.4 g
		Sodium: 36.0 mg
		Gluten Free: Yes
		Lactose Free: Yes
		Vegetarian: Yes
		Vegan: Yes
---
---
Quantity: 1000 g
Ingredient: rice
	Estimated Expiration Date: 2026-04-26
	Estimated Financial Cost per 100g: EUR 0.84
	Nutritional Information:
		Calories: 356.0 kcal
		Carbohydrates: 80.0 g
		Sugar: 0.0 g
		Protein: 6.67 g
		Fat: 0.0 g


In [16]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=preferences,
)

In [17]:
meal_planning_environment.load_recipes_from_json("../recipe_extraction/supplemented_structured_recipes.json")

In [19]:
planner = RandomMealPlanner(meal_planning_environment, random_seed=1)

In [20]:
best_meal_plan = planner.generate_meal_plan()

In [21]:
pantry_consumption_df = planner.get_pantry_consumption()
pantry_consumption_df

,Ingredient,Available,Consumed,Unused,Expires in
0,chicken breast,800,0,800,6d
1,broccoli,1500,0,1500,2d
2,rice,1000,0,1000,2d


In [22]:
meal_plan_df = planner.get_meal_plan_recipes()
meal_plan_df

,Day,Meal 1,Meal 2,Meal 3
0,1,Baguette Toasts,Tarragon-Roasted Halibut with Hazelnut Brown B...,Raspberry Custard
1,2,Roasted Sesame-and Panko-Coated Asparagus with...,Key Lime Pie with Almond Crumb Crust,Sephardic Spinach Patties
2,3,"Root Vegetable ""Lasagna"" with Mushroom Broth",Macaroni and Blue Cheese with Chives,Broccoli Rabe with Pine Nuts and Raisins
3,4,Jícama-Melon Salad,Tropical Fruit Salad,Garnet Yam Puree
4,5,Braised Chicken with Dates and Moroccan Spices,Fried Anchovies with Salbixtada Sauce,"Zucchini, Potato, and Cilantro Soup"
5,6,Pear-Hazelnut Cheesecakes with Pear-Raspberry ...,Spiced Tuna Steaks with Fennel and Red Peppers,Mussels with Coconut Curry Sauce
6,7,Sea Bass with Moroccan Salsa,A Simple Roast Turkey,Roasted Tomato-Chipotle Salsa


In [23]:
shopping_list_df, num_ingredients, total_cost = planner.get_shopping_list()

print(f"Total ingredients needed to purchase: {num_ingredients}")
print(f"Total cost: €{total_cost:.2f}")

shopping_list_df

Total ingredients needed to purchase: 139
Total cost: €177.49


,Ingredient,Quantity to Buy (g),Cost (€)
0,(generous) fennel seeds,1.2,0.01
1,Asian sesame oil,2.2,0.02
2,Coarse kosher salt,100.0,1.00
3,Fresh chives,3.0,0.03
4,Fresh mint,33.3,0.33
...,...,...,...
135,white onion,4.3,0.04
136,whole milk,47.3,0.47
137,zucchini,453.6,4.54
138,zweiback crumbs,14.8,0.15


In [24]:
daily_nutrition_df = planner.get_daily_nutrition()
daily_nutrition_df

,Calories,Protein,Δ Calories and Target Calories,Δ Protein and Target Protein
Day 1,1621.6 kcal,34.1 g,-878.4 kcal,-45.9 g
Day 2,761.7 kcal,20.6 g,-1738.3 kcal,-59.4 g
Day 3,3520.5 kcal,122.4 g,1020.5 kcal,42.4 g
Day 4,3408.0 kcal,32.9 g,908.0 kcal,-47.1 g
Day 5,8673.2 kcal,401.2 g,6173.2 kcal,321.2 g
Day 6,1328.9 kcal,49.9 g,-1171.1 kcal,-30.1 g
Day 7,743.1 kcal,25.2 g,-1756.9 kcal,-54.8 g
